# Prompt Engineering Techniques: Parameter Tuning

Source slide: `slides/prompt-engineering/04.4_Prompt_Engineering_Techniques.Parameter_Tuning.md`

These examples use the real ambient-auth Copilot SDK helper and explain which provider settings to adjust when your Copilot host exposes them.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Temperature

**Failure mode:** Temperature controls randomness. High temperature is helpful for ideation, but it can hurt deterministic extraction tasks by increasing variance.

**Failure test:** The failure run uses an extraction-style prompt that is sensitive to randomness. The real Copilot SDK helper here executes the prompt, while the configuration note explains which temperature to apply in hosts that expose it.

**Technique:** Use low temperature for extraction and high temperature for brainstorming.

**Improved example:** The improved prompt narrows the output shape, and the configuration note tells you to rerun it with temperature=0 for a stable extraction task.

**Configuration note:** For this deterministic classification task, rerun the improved prompt with temperature=0 when your Copilot host exposes temperature controls.


In [ ]:
await run_prompt_pair(
    technique_name='Temperature',
    failure_prompt='Return the single severity label for this incident: “Login failed for all customers in production for 18 minutes.”',
    improved_prompt='Return exactly one severity label from {SEV1, SEV2, SEV3} for this incident: “Login failed for all customers in production for 18 minutes.”',
    failure_test='The failure run uses an extraction-style prompt that is sensitive to randomness. The real Copilot SDK helper here executes the prompt, while the configuration note explains which temperature to apply in hosts that expose it.',
    fix_explanation='The improved prompt narrows the output shape, and the configuration note tells you to rerun it with temperature=0 for a stable extraction task.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
    config_note='For this deterministic classification task, rerun the improved prompt with temperature=0 when your Copilot host exposes temperature controls.',
)


## top_p

**Failure mode:** top_p constrains generation to the most likely token mass. Larger values allow more diversity; smaller values keep outputs tighter and more focused.

**Failure test:** The failure run uses a one-line extraction prompt where overly broad sampling can introduce unnecessary wording drift. The public ambient-auth SDK session API does not expose top_p directly.

**Technique:** Lower top_p when you need focused extraction; raise it only when you actually want diversity.

**Improved example:** The improved prompt constrains the allowed outputs, and the configuration note explains the lower top_p range to try in hosts that expose it.

**Configuration note:** For focused extraction, try top_p around 0.8-0.9 in hosts that expose it, then rerun the improved prompt.


In [ ]:
await run_prompt_pair(
    technique_name='top_p',
    failure_prompt='Read this note and return the primary customer intent: “Can we get SSO added to the business plan next quarter?”',
    improved_prompt='Return exactly one intent label from {question, request, bug} for this note: “Can we get SSO added to the business plan next quarter?”',
    failure_test='The failure run uses a one-line extraction prompt where overly broad sampling can introduce unnecessary wording drift. The public ambient-auth SDK session API does not expose top_p directly.',
    fix_explanation='The improved prompt constrains the allowed outputs, and the configuration note explains the lower top_p range to try in hosts that expose it.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
    config_note='For focused extraction, try top_p around 0.8-0.9 in hosts that expose it, then rerun the improved prompt.',
)


## Seed Values

**Failure mode:** Seeds improve reproducibility so you can compare prompt revisions without random variation masking the difference.

**Failure test:** The failure run uses a creative naming task where repeated runs can vary. The notebook executes the real SDK call, and the configuration note explains when to lock the seed in environments that support it.

**Technique:** Fix the seed during demos, testing, and evaluation when your platform exposes that option.

**Improved example:** The improved prompt adds stronger constraints, and the configuration note explains that pairing it with a fixed seed makes repeated comparisons fairer.

**Configuration note:** When your Copilot host or provider exposes a seed, set one fixed value and rerun the improved prompt to make comparisons reproducible.


In [ ]:
await run_prompt_pair(
    technique_name='Seed Values',
    failure_prompt='Suggest a short launch title for a new authentication experience.',
    improved_prompt='Suggest exactly three short launch titles for a new authentication experience. Each title must be 5 words or fewer and must not use exclamation marks.',
    failure_test='The failure run uses a creative naming task where repeated runs can vary. The notebook executes the real SDK call, and the configuration note explains when to lock the seed in environments that support it.',
    fix_explanation='The improved prompt adds stronger constraints, and the configuration note explains that pairing it with a fixed seed makes repeated comparisons fairer.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
    config_note='When your Copilot host or provider exposes a seed, set one fixed value and rerun the improved prompt to make comparisons reproducible.',
)


## Model-specific Parameters

**Failure mode:** Different providers expose controls like max tokens or stop sequences. Ignoring them can produce answers that are too long or bleed into the wrong format.

**Failure test:** The failure run asks for JSON but gives no format boundary, so the answer may include extra explanation after the data. The real ambient-auth helper cannot set stop sequences directly in the public session API.

**Technique:** Use the model-specific controls your platform exposes alongside prompt instructions.

**Improved example:** The improved prompt asks for JSON only, and the configuration note explains that a stop sequence or output cap should be paired with it when available.

**Configuration note:** If your Copilot host exposes output caps or stop sequences, pair them with the improved prompt to prevent trailing commentary after the JSON.


In [ ]:
await run_prompt_pair(
    technique_name='Model-specific Parameters',
    failure_prompt='Extract the name and email from this text as JSON: “Jamie Rivera can be reached at jamie@example.com.”',
    improved_prompt='Extract the name and email from this text and return JSON only in the form {"name": string, "email": string}: “Jamie Rivera can be reached at jamie@example.com.”',
    failure_test='The failure run asks for JSON but gives no format boundary, so the answer may include extra explanation after the data. The real ambient-auth helper cannot set stop sequences directly in the public session API.',
    fix_explanation='The improved prompt asks for JSON only, and the configuration note explains that a stop sequence or output cap should be paired with it when available.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
    config_note='If your Copilot host exposes output caps or stop sequences, pair them with the improved prompt to prevent trailing commentary after the JSON.',
)
